[Guide Index](README.md)


# 06.03 SPARQL queries as RDF

`starlayer.sparql` can represent SPARQL query syntax as an RDF graph, using the `sqe:` ontology (specific to StarLayer). `sqe:` serializes text into a human-readable RDF syntax tree designed for storage, validation, viewing and editing. The RDF graph can be serialized back as plain text for query execution. 

A corresponding SHACL shapes graph validates the `sqe:` graph and incorporates `shui:` (SHACL UI) components that can be used in a SHACL-based editor of SPARQL queries.

See the [SPARQL guide](03.01-sparql.ipynb) for ordinary query operations. This guide is specifically about treating a *query itself* as an RDF syntax tree you can create, inspect, edit, and validate.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.

In [2]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph

EX = Namespace("http://example.org/")


## Index of functions

SPARQL syntax trees (`sqe:`) in `starlayer.sparql` have three functions.

| Function | What it does | Discussed in |
|---|---|---|
| `sqe_parse_to_tree()` | Parses SPARQL query or update text into an `sqe:` RDF syntax tree graph. | sections 06.03.1, 06.03.1.1, 06.03.1.2 and 06.03.4.1 |
| `sqe_tree_to_text()` | Renders one query of an RDF syntax tree graph back as SPARQL text. | section 06.03.3 |
| `sqe_validate()` | Validates a tree against the SHACL shapes of the `sqe:` ontology, checks that Oxigraph accepts each query, and returns a validation report.| section 06.03.4 |

## 06.03.1 A graph, a query, and the query as RDF

`sqe_parse_to_tree(text_query)` parses SPARQL text into an RDF graph, using the `sqe:` ontology. The graph is a syntax tree of the query with each part (e.g. SELECT, WHERE, FILTER, variables) reflected as an RDF element. Text that is not SPARQL cannot be parsed and raises an error. A query that parses can still be invalid annd get rejected when run.  `sqe_validate()` tests for validation errors.  (see section 06.03.4).

Below, one small data graph and one query are created to be used throughout this guide. The query is first run as ordinary SPARQL text, which finds that `ex:bob` knows `ex:carol`. It is then parsed into an RDF tree, and the tree is output as a graph.

Each ordered part of the tree (the projection, the elements of a WHERE clause, a function's arguments) carries its position as `tree:ord`. A variable or an IRI is held in a `tree:Item`.


In [3]:
from starlayer.sparql import sqe
from starlayer.sparql.vocab import SPEXPR

g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob ex:knows ex:carol ;
        ex:likes ex:dana .
""", format="turtle")

print("data graph:")
print(g.serialize(format="turtle", pretty=True))


# a simple SPARQL query
text_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:knows ?o }"""

# run the query against the data graph, as ordinary SPARQL text
print("query result:")
for row in g.query(text_query):
    print(g.qname(row.s), g.qname(row.o))
print()

# the query is converted to an sqe syntax tree, written into the StarLayerGraph passed in; root is the base node of the query graph.
tree, root = sqe.sqe_parse_to_tree(text_query, StarLayerGraph())

tree.bind("sqe", sqe.SQE)
tree.bind("spexpr", SPEXPR)
tree.bind("tree", "https://github.com/hidden-graph/starlayer/ns/tree#")

print(tree.serialize(pretty=True, only_used=True))

data graph:
@prefix ex: <http://example.org/> .

ex:bob ex:knows ex:carol ;
    ex:likes ex:dana .

query result:
ex:bob ex:carol

@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

_:b0 a sqe:Query, sqe:SelectQuery ;
    sqe:prefixDecl [
        a sqe:PrefixDecl ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:projection [
        a tree:Item ;
        tree:ord 1 ;
        tree:value "o"^^spexpr:Variable
    ], [
        a tree:Item ;
        tree:ord 0 ;
        tree:value "s"^^spexpr:Variable
    ] ;
    sqe:where [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate <http://example.org/knows> ;
        sqe:subject "s"^^spexpr:Variable ;
        tree:ord 0
    ] .



### 06.03.1.1 Giving the query a stable identity with `root=`

`sqe_parse_to_tree()` gives the root node of the tree a fresh blank node by default, as seen above. Passing `root=` gives the query a real, caller-chosen identity instead, so it stays findable by that identity once a graph holds more than one encoded query.

Below, the same query is parsed with `root=` set to `q:bobKnows`, and the tree is printed with that IRI as its root.

In [4]:
from starlayer.graph import NamedNode


query_iri = NamedNode("http://example.org/queries#bobKnows")
named_tree, named_root = sqe.sqe_parse_to_tree(text_query, root=query_iri)

print("root is the given IRI:", named_root.value == query_iri.value)

named_tree.bind("sqe", sqe.SQE)
named_tree.bind("spexpr", SPEXPR)
named_tree.bind("tree", "https://github.com/hidden-graph/starlayer/ns/tree#")
named_tree.bind("q", "http://example.org/queries#")
print(named_tree.serialize(pretty=True, only_used=True))

root is the given IRI: True
@prefix q: <http://example.org/queries#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

q:bobKnows a sqe:Query, sqe:SelectQuery ;
    sqe:prefixDecl [
        a sqe:PrefixDecl ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:projection [
        a tree:Item ;
        tree:ord 0 ;
        tree:value "s"^^spexpr:Variable
    ], [
        a tree:Item ;
        tree:ord 1 ;
        tree:value "o"^^spexpr:Variable
    ] ;
    sqe:where [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate <http://example.org/knows> ;
        sqe:subject "s"^^spexpr:Variable ;
        tree:ord 0
    ] .



### 06.03.1.2 Holding more than one query in a graph

A graph can hold more than one query. When `sqe_parse_to_tree()` is given a graph, it adds the query's tree to that graph, and the root of each query identifies the different queries.

Below, a second query is parsed into the graph that already holds the first one, with its own root IRI. The graph then holds two queries. They are listed by their roots, and each is rendered back to SPARQL text through its root.

In [5]:
from starlayer.graph import RDF

likes_text = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:likes ?o }"""
likes_iri = NamedNode("http://example.org/queries#bobLikes")

# passing a graph that already holds a query adds the new query's tree to it
sqe.sqe_parse_to_tree(likes_text, tree_graph=named_tree, root=likes_iri)

print("queries in the graph:", sorted(named_tree.qname(s) for s in named_tree.subjects(RDF.type, sqe.SQE.Query)))
for root_iri in (query_iri, likes_iri):
    print()
    print(named_tree.qname(root_iri))
    print(sqe.sqe_tree_to_text(named_tree, root_iri))

queries in the graph: ['q:bobKnows', 'q:bobLikes']

q:bobKnows
PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:knows ?o . }

q:bobLikes
PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:likes ?o . }


## 06.03.2 Edit the query by editing the RDF

A query tree is an ordinary RDF graph, so it can be edited with standard graph operations. Edits to the graph structure change the query the tree represents. Annotations that the query does not use, such as a label or a comment, are ignored when the tree is turned back into text (section 06.03.3).

Below, a triple pattern of the tree is found by its predicate `ex:knows` and changed to `ex:likes`. A plain label is added to the query as a whole, and a comment is attached to the query's own `rdf:type` statement as an RDF 1.2 reification, to record why the predicate was changed. The edited tree is printed as a graph.

In [6]:
from starlayer.graph import BlankNode, Literal, RDF, RDFS

# the triple pattern is located and its predicate replaced.
for s, p, o in list(tree.triples((None, sqe.SQE.predicate, EX.knows))):
    tree.remove((s, p, o))
    tree.add((s, p, EX.likes))
    pattern_node = s

# a plain annotation on the query as a whole
tree.add((root, RDFS.label, Literal("bob's likes, not knows")))

# a real RDF 1.2 reification: the query's own rdf:type assertion is annotated
# to record why the predicate of its triple pattern was changed
reifier = BlankNode()
tree.add((reifier, RDFS.comment, Literal("predicate changed from ex:knows to ex:likes")))
tree.add_reification(reifier, (root, RDF.type, sqe.SQE.SelectQuery))

tree.bind("rdfs", RDFS)
print(tree.serialize(pretty=True, only_used=True))

@version "1.2" .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .
@prefix tree: <https://github.com/hidden-graph/starlayer/ns/tree#> .

_:b0 a sqe:Query, sqe:SelectQuery {| rdfs:comment "predicate changed from ex:knows to ex:likes" |} ;
    rdfs:label "bob's likes, not knows" ;
    sqe:prefixDecl [
        a sqe:PrefixDecl ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:projection [
        a tree:Item ;
        tree:ord 0 ;
        tree:value "s"^^spexpr:Variable
    ], [
        a tree:Item ;
        tree:ord 1 ;
        tree:value "o"^^spexpr:Variable
    ] ;
    sqe:where [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate <http://example.org/likes> ;
        sqe:subject "s"^^spexpr:Variable ;
        tree:ord 0

## 06.03.3 Decode back into SPARQL text, and run it

`sqe_tree_to_text()` renders an RDF tree back as SPARQL text, either to run it or to export it. A `StarLayerGraph` runs queries by text, so a tree is run as a query by rendering it to text first. Edits to the tree that do not affect what the query means are ignored. Annotations are not preserved in the text form: to retain them, store the query as a graph, not as text.

Below, the edited tree is rendered to text and the text is printed. The text is then run against the same data graph from section 06.03.1. The result changes from `(bob, carol)` to `(bob, dana)`, reflecting the edit made in section 06.03.2.

In [7]:
# the rdf graph becomes SPARQL text
modified_text = sqe.sqe_tree_to_text(tree, root)
print(modified_text)
print()

# the text is executed by Oxigraph
result = g.query(modified_text)

for row in result:
    print(g.qname(row.s), g.qname(row.o))

PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:likes ?o . }

ex:bob ex:dana


## 06.03.4 sqe Validation

`sqe_validate()` checks the tree against the `sqe:` SHACL shapes. If the tree conforms, it also renders each query in the tree to text and checks that Oxigraph accepts it. This ste catches errors that the SHACL shapes cannot, for example a call to a function that Oxigraph does not support. The tree satisfies the shapes in that case, but the report fails, with a violation on the query's root node.

The shapes used for validation are the registered `sqe_shacl` shapes graph, the file `starlayer/starontology/sqe/sqe_shapes.ttl` of the package (see section 06.06.3 of the [SHACL library registry guide](06.06-shacl-library-registry.ipynb)).

Below, `sqe_validate()` checks the edited tree from section 06.03.2, and it conforms.

In [8]:
# validate the edited syntax tree from above.
report = sqe.sqe_validate(tree)
print("conforms:", report.conforms)

conforms: True


Editing the tree can make the query invalid. Below, the triple pattern's own required `sqe:predicate` is removed, and `sqe_validate()` reports the violation.

In [9]:
# break the tree: remove the triple pattern's own required predicate
pattern_node = next(tree.subjects(RDF.type, sqe.SQE.TriplePattern))
tree.remove((pattern_node, sqe.SQE.predicate, None))

broken_report = sqe.sqe_validate(tree)
print("conforms:", broken_report.conforms)
print(broken_report.report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxCount 1 ; sh:message "sqe:TriplePattern must have exactly one sqe:predicate, an IRI/Variable/property path" ; sh:minCount 1 ; sh:node sqe:PredicateOrVariableOrPathShape ; sh:path sqe:predicate ]
	Focus Node: [ rdf:type sqe:TriplePattern ; sqe:object "o"^^spexpr:Variable ; sqe:subject "s"^^spexpr:Variable ; tree:ord 0 ]
	Result Path: sqe:predicate
	Message: sqe:TriplePattern must have exactly one sqe:predicate, an IRI/Variable/property path



### 06.03.4.1 A query that's valid SPARQL, but fails SHACL validation

A query can be valid SPARQL and still be wrong. For example `SELECT ?s ?z WHERE { ?s ex:knows ?o }` parses and runs as SPARQL without an error. `?z` is never bound by the pattern, so it comes back unbound in every result row. SPARQL itself has no objection to this, but it almost always signals a typo or an incomplete edit: an "output" (`?z` projected) with no matching "input" (nothing in `?s ex:knows ?o` ever binds it). `sqe:SelectQueryShape` carries a SHACL constraint able to catch errors of this nature.

Below, the query runs as ordinary SPARQL and `?z` comes back unbound. The same query is parsed into a tree, which loads as usual, and `sqe_validate()` reports the problem. The validation report is long because it includes the constraint that found the problem: the SPARQL query under "Source Constraint" is the shape's own check over the tree, not the query being validated.

In [12]:
bad_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?z WHERE { ?s ex:knows ?o }"""

# still valid SPARQL - runs fine, ?z just comes back unbound
print("query execution:")
for row in g.query(bad_query):
    print(g.qname(row.s), row.z)
print()

# the query loads as a tree; sqe_validate() reports what is wrong with it
bad_tree, bad_root = sqe.sqe_parse_to_tree(bad_query)
bad_report = sqe.sqe_validate(bad_tree)
print("conforms:", bad_report.conforms)
print(bad_report.report_text)

query execution:
ex:bob None

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:SPARQLConstraintComponent:
	Severity: sh:Violation
	Source Shape: sqe:SelectQueryShape
	Focus Node: [ rdf:type sqe:Query, sqe:SelectQuery ; sqe:prefixDecl [ rdf:type sqe:PrefixDecl ; sqe:namespace <http://example.org/> ; sqe:prefixLabel "ex"^^spexpr:PlainString ] ; sqe:projection [ rdf:type tree:Item ; tree:ord 0 ; tree:value "s"^^spexpr:Variable ], [ rdf:type tree:Item ; tree:ord 1 ; tree:value "z"^^spexpr:Variable ] ; sqe:where [ rdf:type sqe:TriplePattern ; sqe:object "o"^^spexpr:Variable ; sqe:predicate <http://example.org/knows> ; sqe:subject "s"^^spexpr:Variable ; tree:ord 0 ] ]
	Value Node: [ rdf:type sqe:Query, sqe:SelectQuery ; sqe:prefixDecl [ rdf:type sqe:PrefixDecl ; sqe:namespace <http://example.org/> ; sqe:prefixLabel "ex"^^spexpr:PlainString ] ; sqe:projection [ rdf:type tree:Item ; tree:ord 0 ; tree:value "s"^^spexpr:Variable ], [ rdf:type tree:Item ; 